## Spatial Distances

One of the flaw of previous distance analysis is that pmn-other cell distances were greatly influenced by the quanitity of other cells. If there are 100 PMNs and 4 CD4 cells than each CD4 will on average be matched with 25 PMN cells. This inflates distances in general. This new method limits the number of pairs to 1 per cell. So only 4 pairs are possible for the case above

### Load Protein Data

In [2]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np

# Load files on Evan's Laptop
# expr_orig = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\expression.csv", index_col=0)
# expr=expr_orig.transpose()
# metadata = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\metadata.csv", index_col=0)
# umap = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\umap.csv", index_col=0)

#Load files on Lab computer
expr_orig = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\expression.csv", index_col=0)
expr=expr_orig.transpose()
metadata = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\metadata.csv", index_col=0)
umap = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\umap.csv", index_col=0)

# Create AnnData object
adata = sc.AnnData(X=expr.values)

# Assign metadata
adata.obs = metadata
adata.var_names = expr.columns
adata.obs_names = expr.index

# Add spatial coordinates and UMAP to .obsm
# adata.obsm["spatial"] = metadata[['x_FOV_px', 'y_FOV_px']].values  # adjust if needed
adata.obsm["spatial"] = metadata[['x_FOV_px']].assign(y_FOV_px = -metadata['y_FOV_px']).values
adata.obsm["X_umap"] = umap.values

C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import annd

### Creating Distance Matrix

This creates a matrix of the distance of each pmn to each cell of another type.

It will have dimentions of (# PMN cell) * (# Target Cells). In cases of tumor cells or fibroblasts there can be many columns while for NK and Treg there will often be more pmn cells in a sample.

In [ ]:
import my_functions
import pandas as pd

sample_id="c_1_1"
target_cell="Macrophages"

# Gets Distance Matrix Between PMNs and Target Cells
results=my_functions.nearest_cells_of_particular_type(adata,sample_id,target_cell)


distance_matrix=results["Distance Matrix"]
distance_matrix=distance_matrix.drop(["Minimum Distance","Average Distance"],axis=1)
display(distance_matrix)

num_pmns,num_target=distance_matrix.shape
display(num_pmns)
display(num_target)
#Create Lists of rows and columns
target_cell_list=distance_matrix.columns.tolist()
pmn_cell_list=distance_matrix.index.tolist()



In [32]:
import numpy as np
from scipy.optimize import linear_sum_assignment

# --- 1. Define your Distance Matrix ---
# Let's assume you have 5 PMN cells (rows) and 3 macrophages (columns).
# The values represent the distance between each PMN and macrophage.
distance_matrix = np.array([
    [82, 83, 69],  # PMN 1 to Macrophages 1, 2, 3
    [77, 37, 49],  # PMN 2 to Macrophages 1, 2, 3
    [11, 69,  5],  # PMN 3 to Macrophages 1, 2, 3
    [74, 92, 35],  # PMN 4 to Macrophages 1, 2, 3
    [  8, 9, 98]   # PMN 5 to Macrophages 1, 2, 3
])

# --- 2. Apply the Hungarian Algorithm ---
# The linear_sum_assignment function will find the optimal assignment
# that minimizes the sum of the distances.
pmn_indices, macrophage_indices = linear_sum_assignment(distance_matrix)

# --- 3. Extract the Results ---
# The function returns the optimal row (PMN) and column (macrophage) indices.
optimal_pairs = list(zip(pmn_indices, macrophage_indices))

# Calculate the minimum total distance
min_total_distance = distance_matrix[pmn_indices, macrophage_indices].sum()

# --- 4. Display the Optimal Pairings and Total Distance ---
print("Optimal PMN-Macrophage Pairs (PMN index, Macrophage index):")
for pmn_idx, mac_idx in optimal_pairs:
    distance = distance_matrix[pmn_idx, mac_idx]
    print(f"  PMN {pmn_idx} is paired with Macrophage {mac_idx} (Distance: {distance})")

print(f"\nMinimum Overall Distance: {min_total_distance}")

# Note on unpaired PMNs:
all_pmn_indices = set(range(distance_matrix.shape[0]))
paired_pmn_indices = set(pmn_indices)
unpaired_pmn_indices = all_pmn_indices - paired_pmn_indices

print("\nUnpaired PMN cells (by index):")
print(f"  {list(unpaired_pmn_indices)}")

Optimal PMN-Macrophage Pairs (PMN index, Macrophage index):
  PMN 1 is paired with Macrophage 1 (Distance: 37)
  PMN 2 is paired with Macrophage 2 (Distance: 5)
  PMN 4 is paired with Macrophage 0 (Distance: 8)

Minimum Overall Distance: 50

Unpaired PMN cells (by index):
  [0, 3]


In [6]:
import my_functions
def mininum_distance_exclusive_pairing(distance_matrix):
    import pandas as pd
    import numpy as np
    from scipy.optimize import linear_sum_assignment
    
    # --- 1. Define your Distance Matrix as a Pandas DataFrame ---
    distance_data=distance_matrix
    distance_df = pd.DataFrame(distance_data)
    
    # --- 2. Apply the Hungarian Algorithm ---
    # The scipy function requires a NumPy array, so we extract it using .to_numpy()
    cost_matrix = distance_df.to_numpy()
    pmn_indices, target_indices = linear_sum_assignment(cost_matrix)
    
    # --- 3. Map Indices back to DataFrame Labels ---
    # Get the actual labels from the DataFrame's index and columns
    paired_pmns = distance_df.index[pmn_indices]
    paired_targets = distance_df.columns[target_indices]
    
    # Get the distances for the optimal pairs
    optimal_distances = cost_matrix[pmn_indices, target_indices]
    
    # --- 4. Create a DataFrame for the Results and Display ---
    # This provides a clean, readable output of the optimal pairings.
    results_df = pd.DataFrame({
        'PMN_Cell': paired_pmns,
        'Paired_Target': paired_targets,
        'Distance': optimal_distances
    })
    
    average_distance = results_df['Distance'].mean()
    print(f'The average distance is: {average_distance}')
    
    # Calculate the minimum total distance
    min_total_distance = optimal_distances.sum()
    print(f"\nMinimum Overall Distance: {min_total_distance}")
    
    # Identify unpaired PMNs using the DataFrame's index
    all_pmns = set(distance_df.index)
    paired_pmns_set = set(paired_pmns)
    unpaired_pmns = all_pmns - paired_pmns_set

    return{
        "Pair Matrix":results_df,
        "Average Distance":average_distance
    }

def nearest_cells_of_particular_type(adata,sample_ID,cell_distance_type):
    """
    Calculates distance matrix between PMNs and target cells of a particular type,
    along with summary statistics.

    Parameters:
    -----------
    adata : AnnData
        Annotated data object containing spatial information for cells.
    sample_ID : str
        Identifier for the specific sample to analyze.
    cell_distance_type : str
        Cell type to use as the target for distance calculations (e.g., "CD4+T_cells").

    Returns:
    --------
    dict
        A dictionary containing:
        - 'Distance Matrix' (pd.DataFrame): Matrix of distances between PMNs and target cells,
          including additional columns for 'Minimum Distance' and 'Average Distance' per PMN.
        - 'Average Distance' (float): Overall average distance across the entire matrix.
        - 'Average Mininum Distance' (float): Average of each PMN's minimum distance to target cells.
    """
    import pandas as pd
    import my_functions

    # Creates lists of cell types for PMN and the target comparison
    pmn_list=my_functions.pmn_counter(adata,sample_ID)['PMN Names']
    target_cell_list=my_functions.matching_cell_list(adata,sample_ID,cell_distance_type)['Cell Names']

    pmn_count=my_functions.pmn_counter(adata,sample_ID)['PMN Count']
    target_cell_count=my_functions.matching_cell_list(adata,sample_ID,cell_distance_type)['Cell Count']

    print(f"The pmn count is {pmn_count} the other cell count is {target_cell_count}")
    
    # Initialize results storage: one column for this cell type
    distance_matrix = pd.DataFrame(index=pmn_list, columns=target_cell_list)
    
    for pmn in pmn_list:
        for target_cell in target_cell_list:
            distance_matrix.loc[pmn,target_cell]=my_functions.find_distance(adata,pmn, target_cell)["Distance um"]
    
    # Ensure numeric types
    distance_matrix = distance_matrix.apply(pd.to_numeric, errors='coerce')
    
    # Calculate overall average distance across entire matrix
    overall_avg_distance = distance_matrix.mean().mean()

    # Calculate average mininum distance
    average_min_distance = distance_matrix.min(axis=1).mean()
            
    # Add columns first (unrounded)
    distance_matrix['Minimum Distance'] = distance_matrix.min(axis=1).round(2)
    distance_matrix['Average Distance'] = distance_matrix.mean(axis=1).round(2)
    
    return{
        "Distance Matrix":distance_matrix,
        "Average Distance":overall_avg_distance,
        "Average Mininum Distance": average_min_distance
    }


sample_id="c_3_7"
target_cell="CD4+T_cells"

# Gets Distance Matrix Between PMNs and Target Cells
results=my_functions.nearest_cells_of_particular_type(adata,sample_id,target_cell)
distance_matrix=results["Distance Matrix"]
distance_matrix=distance_matrix.drop(["Minimum Distance","Average Distance"],axis=1)

results=mininum_distance_exclusive_pairing(distance_matrix)
display(results["Pair Matrix"])

results2=nearest_cells_of_particular_type(adata,sample_id,target_cell)
display(results2["Average Mininum Distance"])
display(results2["Distance Matrix"])


The average distance is: 44.02448729446192

Minimum Overall Distance: 528.293847533543


,PMN_Cell,Paired_Target,Distance
0,c_3_7_4,c_3_7_5,13.780097
1,c_3_7_50,c_3_7_16,74.059804
2,c_3_7_234,c_3_7_2587,9.767145
3,c_3_7_260,c_3_7_2651,33.739570
4,c_3_7_426,c_3_7_744,98.981659
5,c_3_7_896,c_3_7_2912,50.241270
6,c_3_7_1271,c_3_7_926,98.596680
7,c_3_7_1847,c_3_7_1766,29.083901
8,c_3_7_2611,c_3_7_2623,12.157676
9,c_3_7_2843,c_3_7_844,26.633697


The pmn count is 127 the other cell count is 12


145.79226907872894

,c_3_7_5,c_3_7_16,c_3_7_744,c_3_7_844,c_3_7_926,c_3_7_1305,c_3_7_1766,c_3_7_2152,c_3_7_2587,c_3_7_2623,c_3_7_2651,c_3_7_2912,Minimum Distance,Average Distance
c_3_7_3,107.185185,407.400035,480.594326,264.485842,331.569186,391.787022,619.987151,693.845858,106.189955,108.270233,428.730108,456.475168,106.19,346.36
c_3_7_4,13.780097,313.992180,398.605978,247.800513,290.660441,415.998430,655.405811,717.272591,191.045623,181.223727,338.863371,392.092421,13.78,320.81
c_3_7_10,67.207560,233.016242,331.928293,260.213833,275.837225,451.106028,694.172524,745.750865,269.050308,255.209253,262.641660,346.406060,67.21,327.67
c_3_7_11,76.106967,224.112000,325.255853,263.453409,276.024051,456.028256,699.221813,749.723035,277.818212,263.728217,254.605372,342.545244,76.11,329.59
c_3_7_29,24.366951,276.573926,364.189166,245.379227,276.038775,426.156630,667.977634,724.739188,225.611991,212.861370,301.609975,365.287210,24.37,318.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
c_3_7_3321,748.553783,656.884060,446.940943,554.668989,474.440596,591.142252,667.629738,600.064605,816.042703,781.630269,553.538203,372.769551,372.77,587.47
c_3_7_3373,729.179320,696.871857,494.653541,505.712845,442.645092,495.173381,536.716118,461.633248,760.755218,726.478195,595.751901,388.147231,388.15,555.53
c_3_7_3409,819.298090,738.736652,528.977820,614.321812,539.574495,627.959033,675.114245,596.248189,874.500436,840.020580,635.385728,449.488262,449.49,645.32
c_3_7_3414,779.476442,739.407699,534.620348,556.835943,493.204997,542.522465,570.833469,489.252374,811.704305,777.447750,637.452927,433.281673,433.28,599.95
